# Extract NYC EMS Incident Data

This file uses the City of New York Data Portal's API to extract the NYC EMS Incident Dispatch Data from 01-01-2023 to 12-31-2024. The data are transformed from a JSON structure to a Pandas dataframe structure, cleaned, and then exported to a CSV.

In [ ]:
! pip install sodapy

In [ ]:
# import libraries
import pandas as pd
from sodapy import Socrata
from google.colab import userdata
from google.colab import files

In [ ]:
# define API user data
token = userdata.get('SOCRATA_APP_TOKEN')
user = userdata.get('SOCRATA_USERNAME')
user_pass = userdata.get('SOCRATA_PASSWORD')

# initialize Socrata client
client = Socrata("data.cityofnewyork.us",
                 app_token=token,
                 username=user,
                 password=user_pass,
                 timeout=60)

In [ ]:
# define variables
offset = 0
limit = 50000
all_records = []
start_date = '2023-01-01T00:00:00'
end_date = '2024-12-31T23:59:59'
where_clause = f"INCIDENT_DATETIME between '{start_date}' and '{end_date}'"

In [ ]:
# fetch data with limit and offset parameters

while True:

  # return results as JSON from API / converted to Python list of dictionaries by sodapy
  results = client.get("76xm-jjuj", limit=limit, offset=offset, where=where_clause)

  # convert to pandas DataFrame
  chunk_df = pd.DataFrame.from_records(results)

  # Check if the chunk is empty, if so, break the loop
  if chunk_df.empty:
    print(f"No more records to fetch. Breaking loop at offset {offset}.")
    break

  # Append the chunk to the all_records list
  all_records.append(chunk_df)

  # Increment the offset
  offset += limit

  # Calculate total records fetched so far
  total_records_fetched = sum(len(df) for df in all_records)

  # Print progress message
  print(f"Current offset: {offset}.")

Current offset: 50000.
Current offset: 100000.
Current offset: 150000.
Current offset: 200000.
Current offset: 250000.
Current offset: 300000.
Current offset: 350000.
Current offset: 400000.
Current offset: 450000.
Current offset: 500000.
Current offset: 550000.
Current offset: 600000.
Current offset: 650000.
Current offset: 700000.
Current offset: 750000.
Current offset: 800000.
Current offset: 850000.
Current offset: 900000.
Current offset: 950000.
Current offset: 1000000.
Current offset: 1050000.
Current offset: 1100000.
Current offset: 1150000.
Current offset: 1200000.
Current offset: 1250000.
Current offset: 1300000.
Current offset: 1350000.
Current offset: 1400000.
Current offset: 1450000.
Current offset: 1500000.
Current offset: 1550000.
Current offset: 1600000.
Current offset: 1650000.
Current offset: 1700000.
Current offset: 1750000.
Current offset: 1800000.
Current offset: 1850000.
Current offset: 1900000.
Current offset: 1950000.
Current offset: 2000000.
Current offset: 2050

In [ ]:
# create dataframe from data
raw_df = pd.concat(all_records, ignore_index=True)
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3192252 entries, 0 to 3192251
Data columns (total 31 columns):
 #   Column                          Dtype 
---  ------                          ----- 
 0   cad_incident_id                 object
 1   incident_datetime               object
 2   initial_call_type               object
 3   initial_severity_level_code     object
 4   final_call_type                 object
 5   final_severity_level_code       object
 6   first_assignment_datetime       object
 7   valid_dispatch_rspns_time_indc  object
 8   dispatch_response_seconds_qy    object
 9   first_activation_datetime       object
 10  first_on_scene_datetime         object
 11  valid_incident_rspns_time_indc  object
 12  incident_response_seconds_qy    object
 13  incident_travel_tm_seconds_qy   object
 14  incident_close_datetime         object
 15  held_indicator                  object
 16  incident_disposition_code       object
 17  borough                         object
 18  in

In [ ]:
# drop rows where VALID_DISPATCH_RSPNS_TIME_INDC or VALID_INCIDENT_RSPNS_TIME_INDC is 'N'
df_dropped = raw_df[~((raw_df['valid_dispatch_rspns_time_indc'] == 'N') | (raw_df['valid_incident_rspns_time_indc'] == 'N'))]

# drop rows where REOPEN_INDICATOR, STANDBY_INDICATOR, or TRANSFER_INDICATOR is 'Y'
df_dropped = df_dropped[~((df_dropped['held_indicator'] == 'Y') |
                          (df_dropped['reopen_indicator'] == 'Y') |
                          (df_dropped['standby_indicator'] == 'Y') |
                          (df_dropped['transfer_indicator'] == 'Y'))]

# drop rows where INITIAL_CALL_TYPE is 'UNKNOWN'
df_dropped = df_dropped[~(df_dropped['initial_call_type'] == 'UNKNOWN')]

# define columns to drop columns
cols_to_drop = ['first_activation_datetime',
                'first_to_hosp_datetime',
                'first_hosp_arrival_datetime',
                'held_indicator',
                'incident_disposition_code',
                'policeprecinct',
                'citycouncildistrict',
                'communitydistrict',
                'communityschooldistrict',
                'congressionaldistrict',
                'valid_dispatch_rspns_time_indc',
                'valid_incident_rspns_time_indc',
                'reopen_indicator',
                'standby_indicator',
                'transfer_indicator']

# drop columns
df = df_dropped.drop(columns=cols_to_drop)

In [ ]:
# convert columns to datetime object in final_df
df['incident_datetime'] = pd.to_datetime(df['incident_datetime'])
df['first_assignment_datetime'] = pd.to_datetime(df['first_assignment_datetime'])
df['first_on_scene_datetime'] = pd.to_datetime(df['first_on_scene_datetime'])
df['incident_close_datetime'] = pd.to_datetime(df['incident_close_datetime'])

In [ ]:
# integer columns
int_cols = ['cad_incident_id',
            'initial_severity_level_code',
            'final_severity_level_code',
            'dispatch_response_seconds_qy',
            'incident_response_seconds_qy',
            'incident_travel_tm_seconds_qy',
            'zipcode']

# convert columns to numeric
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# drop rows with missing values
df = df.dropna()

# convert the cleaned columns to integer data type
df[int_cols] = df[int_cols].astype(int)

In [ ]:
# check dataframe info
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2529402 entries, 2 to 3192250
Data columns (total 16 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   cad_incident_id                int64         
 1   incident_datetime              datetime64[ns]
 2   initial_call_type              object        
 3   initial_severity_level_code    int64         
 4   final_call_type                object        
 5   final_severity_level_code      int64         
 6   first_assignment_datetime      datetime64[ns]
 7   dispatch_response_seconds_qy   int64         
 8   first_on_scene_datetime        datetime64[ns]
 9   incident_response_seconds_qy   int64         
 10  incident_travel_tm_seconds_qy  int64         
 11  incident_close_datetime        datetime64[ns]
 12  borough                        object        
 13  incident_dispatch_area         object        
 14  zipcode                        int64         
 15  special_event_indica

In [ ]:
# check range of data retrieved
print(f'First Date: {min(df['incident_datetime'])}')
print(f'Last Date: {max(df['incident_datetime'])}')

First Date: 2023-01-01 00:00:30
Last Date: 2024-12-31 23:59:48


In [ ]:
df.to_csv('nyc_ems_2023_2024_data.csv', index=False)
files.download('nyc_ems_2023_2024_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test importing data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
ems_data = pd.read_csv('/content/drive/MyDrive/Academics/Y3_Fa25/I320D_ML/final-project/nyc_ems_2023_2024_data.csv')

In [ ]:
ems_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2529402 entries, 0 to 2529401
Data columns (total 16 columns):
 #   Column                         Dtype 
---  ------                         ----- 
 0   cad_incident_id                int64 
 1   incident_datetime              object
 2   initial_call_type              object
 3   initial_severity_level_code    int64 
 4   final_call_type                object
 5   final_severity_level_code      int64 
 6   first_assignment_datetime      object
 7   dispatch_response_seconds_qy   int64 
 8   first_on_scene_datetime        object
 9   incident_response_seconds_qy   int64 
 10  incident_travel_tm_seconds_qy  int64 
 11  incident_close_datetime        object
 12  borough                        object
 13  incident_dispatch_area         object
 14  zipcode                        int64 
 15  special_event_indicator        object
dtypes: int64(7), object(9)
memory usage: 308.8+ MB
